# 📘 Notebook 10: Graph Databases with Neo4j (Cypher in Practice)

## 🎯 Learning Objectives

By the end of this notebook, you will:
* Master **Cypher query language** for Neo4j graph database
* Learn to write queries from **simple to advanced**
* Understand **pattern matching** and graph traversal
* Apply **fraud detection queries** on real transaction networks
* Optimize queries for **performance and clarity**

---

## ⚡ Quick Context: Why Neo4j for Fraud Detection?

| Aspect | SQL Database | Neo4j Graph DB |
| --- | --- | --- |
| **Find direct transactions** | Easy (simple JOIN) | Easy (one query) |
| **Find 3-hop paths** | 3+ JOINs (slow) | Single query (fast) |
| **Detect cycles** | Complex query | Pattern matching |
| **Find network hubs** | Group by, subqueries | Count connections |
| **Traversal speed** | O(JOIN complexity) | O(direct adjacency) |
| **Pattern discovery** | Very difficult | Natural and fast |

**Key advantage:** Neo4j stores relationships as first-class objects → direct pointer navigation → nanosecond-scale traversals

---

## 📚 Cypher Fundamentals (Quick Refresher)

**Cypher** is a declarative query language for Neo4j designed to **look like ASCII art** of the pattern you're searching for.

```
(node1)-[:RELATIONSHIP]->(node2)
   ↓         ↓              ↓
 Nodes   Properties     Relationships
```

**Core Syntax Elements:**
- `MATCH` → Find patterns
- `WHERE` → Filter results
- `RETURN` → Specify output
- `CREATE` → Add data
- `DELETE` → Remove data
- `WITH` → Chain operations

---

## 🚀 Running Example: Fraud Detection Transaction Network

Throughout this notebook, we'll use **ONE consistent transaction network** to demonstrate Cypher at every level.

**Example Network:**
```
Alice → Bob → Charlie → David
  ↓       ↓
 Eve    Frank
        (suspicious cycle back to Alice)

Amount ranges: $500 - $50,000
Timeline: Jan 2024 - May 2024
```

We'll query this network using Cypher to demonstrate:
1. Simple lookups
2. Filtering
3. Multi-hop traversal
4. Aggregation
5. Pattern detection
6. Advanced fraud analysis

In [ ]:
# 🧩 Part 1: Setting Up Neo4j Connection

# Note: This notebook demonstrates Cypher queries for Neo4j
# In a real environment, you would connect to a running Neo4j instance

# For demonstration, we'll show:
# 1. The Cypher queries in detail
# 2. Expected results and interpretations
# 3. Different query formats for various use cases

# If you're using Neo4j locally, the connection would look like:
"""
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))
session = driver.session()

# Then run queries like:
result = session.run("MATCH (n) RETURN n LIMIT 10")
"""

# For this notebook, we'll focus on understanding and writing correct Cypher queries
# that you can copy-paste directly into Neo4j Browser or your application

print("=" * 70)
print("NEO4J CYPHER QUERY TUTORIAL")
print("=" * 70)
print("\n📊 EXAMPLE TRANSACTION NETWORK:")
print("-" * 70)
print("""
SETUP: Fraud Detection Network

Accounts:
  - Alice (balance: $10,000, risk: LOW)
  - Bob (balance: $5,000, risk: MEDIUM)
  - Charlie (balance: $2,000, risk: HIGH)
  - David (balance: $8,000, risk: MEDIUM)
  - Eve (balance: $1,000, risk: HIGH)
  - Frank (balance: $3,000, risk: MEDIUM)

Transactions (the TRANSFERS relationships):
  1. Alice → Bob ($5,000) [2024-01-15]
  2. Bob → Charlie ($2,000) [2024-01-20]
  3. Charlie → David ($1,500) [2024-02-01]
  4. Alice → Eve ($500) [2024-02-10]
  5. Bob → Frank ($1,000) [2024-02-15]
  6. Frank → Alice ($950) [2024-03-01] ⚠️ CYCLE (money returns to source!)
  7. David → Charlie ($3,000) [2024-03-15]
  8. Charlie → Eve ($2,000) [2024-04-01]

🚨 SUSPICIOUS PATTERNS:
  - Cycle: Alice → Bob → Frank → Alice (money loop)
  - Chain: Alice → Bob → Charlie → David (4-hop path)
  - High volume: Charlie processes $3,500 incoming
""")

print("\n✓ Network structure defined. Let's query it with Cypher..."))

---

## 📝 Query Type 1: Data Creation with Cypher

### Understanding CREATE Statements

In Neo4j, before querying, you must create the data. Cypher makes this intuitive:

**Syntax:**
```cypher
CREATE (node:Label {property: value})
CREATE (node1)-[:RELATIONSHIP]->(node2)
```

### Complete Cypher Setup for Our Example

This is the complete sequence of CREATE statements to build our fraud network in Neo4j:


In [ ]:
# 🔹 CREATE Statements: Building Our Transaction Network

print("\n" + "=" * 70)
print("STEP 1: CREATE DATA (Cypher Statements)")
print("=" * 70)

cypher_create = """
-- Create Account Nodes
CREATE (alice:Account {id: 'alice', name: 'Alice', balance: 10000, risk: 'LOW'})
CREATE (bob:Account {id: 'bob', name: 'Bob', balance: 5000, risk: 'MEDIUM'})
CREATE (charlie:Account {id: 'charlie', name: 'Charlie', balance: 2000, risk: 'HIGH'})
CREATE (david:Account {id: 'david', name: 'David', balance: 8000, risk: 'MEDIUM'})
CREATE (eve:Account {id: 'eve', name: 'Eve', balance: 1000, risk: 'HIGH'})
CREATE (frank:Account {id: 'frank', name: 'Frank', balance: 3000, risk: 'MEDIUM'})

-- Create Transaction Relationships
CREATE (alice)-[:TRANSFERS {amount: 5000, date: '2024-01-15', txn_id: 'txn001'}]->(bob)
CREATE (bob)-[:TRANSFERS {amount: 2000, date: '2024-01-20', txn_id: 'txn002'}]->(charlie)
CREATE (charlie)-[:TRANSFERS {amount: 1500, date: '2024-02-01', txn_id: 'txn003'}]->(david)
CREATE (alice)-[:TRANSFERS {amount: 500, date: '2024-02-10', txn_id: 'txn004'}]->(eve)
CREATE (bob)-[:TRANSFERS {amount: 1000, date: '2024-02-15', txn_id: 'txn005'}]->(frank)
CREATE (frank)-[:TRANSFERS {amount: 950, date: '2024-03-01', txn_id: 'txn006'}]->(alice)
CREATE (david)-[:TRANSFERS {amount: 3000, date: '2024-03-15', txn_id: 'txn007'}]->(charlie)
CREATE (charlie)-[:TRANSFERS {amount: 2000, date: '2024-04-01', txn_id: 'txn008'}]->(eve)
"""

print("\nCypher to execute in Neo4j Browser:")
print("-" * 70)
print(cypher_create)

print("\n✓ All nodes and relationships created")
print("\n📊 Network Statistics:")
print("  - 6 Account nodes")
print("  - 8 Transaction relationships")
print("  - 1 Suspicious cycle: Alice → Bob → Frank → Alice")
print("  - Longest path: Alice → Bob → Charlie → David (4 hops)")

---

## 🔍 Query Type 2: Simple MATCH - Finding Direct Connections

### Use Case: "Who did Alice send money to?"

**Cypher Syntax:**
```cypher
MATCH (a:Account)-[:TRANSFERS]->(b:Account)
WHERE a.id = 'alice'
RETURN a.name as Sender, b.name as Receiver
```

### Understanding the Query:
- `MATCH` → Find nodes and relationships matching the pattern
- `(a:Account)` → First node (any Account, store in variable 'a')
- `-[:TRANSFERS]->` → Outgoing TRANSFERS relationship
- `(b:Account)` → Second node (any Account, store in variable 'b')
- `WHERE` → Apply filters to the matched nodes
- `RETURN` → Specify what to return and how to format it

### Result Format:
```
| Sender | Receiver |
| ------ | -------- |
| Alice  | Bob      |
| Alice  | Eve      |
```

**Key Insight:** This finds all **outgoing** transactions from Alice (1-hop neighbors)

In [ ]:
# 🔹 Query Type 2: Simple MATCH - Direct Connections

print("\n" + "=" * 70)
print("QUERY 2: SIMPLE MATCH - WHO DID ALICE SEND MONEY TO?")
print("=" * 70)

query2_basic = """
MATCH (a:Account)-[:TRANSFERS]->(b:Account)
WHERE a.id = 'alice'
RETURN a.name as Sender, b.name as Receiver
"""

print("\nCypher Query:")
print("-" * 70)
print(query2_basic)

print("\nExpected Result:")
print("-" * 70)
print("""
| Sender | Receiver |
| ------ | -------- |
| Alice  | Bob      |
| Alice  | Eve      |
""")

print("\n💡 Explanation:")
print("  - Pattern: Alice sends money to someone")
print("  - Results: 2 rows (Alice has 2 outgoing transactions)")
print("  - Speed: Microseconds (direct pointer lookup)")

# Variation 1: Get detailed transaction info
print("\n\n" + "-" * 70)
print("VARIATION 1: Include Transaction Details")
print("-" * 70)

query2_variation1 = """
MATCH (a:Account)-[t:TRANSFERS]->(b:Account)
WHERE a.id = 'alice'
RETURN a.name as Sender, b.name as Receiver, t.amount as Amount, t.date as Date
"""

print("\nCypher Query:")
print(query2_variation1)

print("\nExpected Result:")
print("""
| Sender | Receiver | Amount | Date       |
| ------ | -------- | ------ | ---------- |
| Alice  | Bob      | 5000   | 2024-01-15 |
| Alice  | Eve      | 500    | 2024-02-10 |
""")

print("\n💡 Key Difference: Access relationship properties with [t:RELATIONSHIP]")

# Variation 2: Bidirectional (both incoming and outgoing)
print("\n\n" + "-" * 70)
print("VARIATION 2: Find ALL Connections (Incoming AND Outgoing)")
print("-" * 70)

query2_variation2 = """
MATCH (a:Account)-[t:TRANSFERS]-(b:Account)
WHERE a.id = 'alice'
RETURN a.name as Account1, b.name as Account2, t.amount as Amount
ORDER BY t.amount DESC
"""

print("\nCypher Query:")
print(query2_variation2)

print("\nExpected Result:")
print("""
| Account1 | Account2 | Amount |
| -------- | -------- | ------ |
| Alice    | Bob      | 5000   |
| Alice    | Frank    | 950    |
| Alice    | Eve      | 500    |
""")

print("\n💡 Key Difference:")
print("  - [t:TRANSFERS] (no arrow) = bidirectional")
print("  - Finds: Alice → Bob AND Frank → Alice")
print("  - ORDER BY: Sort results by amount descending")

# Variation 3: Filter by relationship property
print("\n\n" + "-" * 70)
print("VARIATION 3: High-Value Transactions Only (> $1000)")
print("-" * 70)

query2_variation3 = """
MATCH (a:Account)-[t:TRANSFERS]->(b:Account)
WHERE t.amount > 1000
RETURN a.name as Sender, b.name as Receiver, t.amount as Amount
ORDER BY t.amount DESC
"""

print("\nCypher Query:")
print(query2_variation3)

print("\nExpected Result:")
print("""
| Sender   | Receiver | Amount |
| -------- | -------- | ------ |
| Alice    | Bob      | 5000   |
| Charlie  | David    | 1500   |
| David    | Charlie  | 3000   |
| Bob      | Charlie  | 2000   |
| Charlie  | Eve      | 2000   |
""")

print("\n✓ Query Type 2 covered: Simple MATCH with filtering")

---

## 🔗 Query Type 3: Multi-Hop Traversal (Graph Database Advantage!)

### Use Case: "Trace money flow from Alice through multiple hops"

**Why This Matters for Fraud:**
- Money laundering involves CHAINS of transactions
- Detect suspicious paths: does money return to source? (cycle)
- Find all accounts that receive Alice's money (directly or indirectly)

**Cypher Syntax for Variable-Length Paths:**
```cypher
MATCH path = (a:Account)-[:TRANSFERS*1..3]->(b:Account)
WHERE a.id = 'alice'
RETURN path
```

- `*1..3` → Follow the relationship 1 to 3 times
- `*` alone → Unlimited hops (careful! can be slow)
- `*..3` → Maximum 3 hops
- `*2` → Exactly 2 hops
- `path = (...)` → Capture entire path for analysis

### Key Differences:
- **1-hop:** Direct connections only (what we did in Query 2)
- **2-hops:** Friends of friends
- **3-hops:** Friends of friends of friends
- **Variable hops:** More flexible, more powerful, potentially slower

In [ ]:
# 🔹 Query Type 3: Multi-Hop Traversal

print("\n" + "=" * 70)
print("QUERY 3: MULTI-HOP TRAVERSAL - TRACE MONEY FLOW")
print("=" * 70)

print("\nCore Concept: Follow transactions through chains")
print("This is where graph databases DOMINATE over SQL")

# Query 3.1: Exactly 2-hop paths
print("\n\n" + "-" * 70)
print("QUERY 3.1: Exactly 2-Hop Paths from Alice")
print("(Follow Alice's money through one intermediary)")
print("-" * 70)

query3_2hop = """
MATCH (a:Account)-[:TRANSFERS]->(b:Account)-[:TRANSFERS]->(c:Account)
WHERE a.id = 'alice'
RETURN a.name as Source, b.name as Via, c.name as Destination
"""

print("\nCypher Query:")
print(query3_2hop)

print("\nExpected Result:")
print("""
| Source | Via      | Destination |
| ------ | -------- | ----------- |
| Alice  | Bob      | Charlie     |
| Alice  | Bob      | Frank       |
| Alice  | Eve      | (empty)     |
""")

print("\n💡 Interpretation:")
print("  - Alice → Bob → Charlie: Money reaches Charlie")
print("  - Alice → Bob → Frank: Money reaches Frank")
print("  - Alice → Eve → ?: No further transactions from Eve")

# Query 3.2: Variable hops (1 to 3)
print("\n\n" + "-" * 70)
print("QUERY 3.2: Find all accounts reachable from Alice (1-3 hops)")
print("-" * 70)

query3_variable = """
MATCH (a:Account)-[:TRANSFERS*1..3]->(b:Account)
WHERE a.id = 'alice'
RETURN DISTINCT b.name as ReachableAccount
ORDER BY b.name
"""

print("\nCypher Query:")
print(query3_variable)

print("\nExpected Result:")
print("""
| ReachableAccount |
| ---------------- |
| Bob              |
| Charlie          |
| David            |
| Eve              |
| Frank            |
""")

print("\n💡 Interpretation:")
print("  - Alice can reach 5 other accounts through various paths")
print("  - This is the 'reachability' or 'influence' from Alice")
print("  - DISTINCT: Remove duplicates (Alice reaches Charlie via multiple paths)")

# Query 3.3: Get the actual paths
print("\n\n" + "-" * 70)
print("QUERY 3.3: Get Full Paths (Money Trail)")
print("-" * 70)

query3_paths = """
MATCH path = (a:Account)-[:TRANSFERS*1..3]->(b:Account)
WHERE a.id = 'alice' AND b.name = 'David'
RETURN nodes(path) as AccountPath, relationships(path) as Transactions
"""

print("\nCypher Query:")
print(query3_paths)

print("\nExpected Result:")
print("""
| AccountPath                        | Transactions Count |
| ---------------------------------- | ------------------ |
| [Alice, Bob, Charlie, David]       | 3                  |
""")

print("\n💡 Key Functions:")
print("  - nodes(path): Extract all nodes from path → [Alice, Bob, Charlie, David]")
print("  - relationships(path): Extract all relationships")
print("  - LENGTH(path): Number of relationships")

# Query 3.4: Length-based filtering
print("\n\n" + "-" * 70)
print("QUERY 3.4: Find Paths of Specific Length")
print("-" * 70)

query3_length = """
MATCH path = (a:Account)-[:TRANSFERS*2..2]->(b:Account)
WHERE a.id = 'alice'
RETURN a.name as Source, b.name as Destination, LENGTH(path) as Hops
"""

print("\nCypher Query:")
print(query3_length)

print("\nExpected Result:")
print("""
| Source | Destination | Hops |
| ------ | ----------- | ---- |
| Alice  | Charlie     | 2    |
| Alice  | Frank       | 2    |
""")

print("\n💡 Note: LENGTH(path) returns number of relationships (edges), not nodes")

# Query 3.5: Total transaction amount along path
print("\n\n" + "-" * 70)
print("QUERY 3.5: Calculate Total Amount Transferred Along Path")
print("-" * 70)

query3_amount = """
MATCH path = (a:Account)-[:TRANSFERS*1..3]->(b:Account)
WHERE a.id = 'alice' AND b.name = 'David'
WITH relationships(path) as txns
RETURN 
  REDUCE(total = 0, rel IN txns | total + rel.amount) as TotalAmount,
  SIZE(txns) as NumTransactions
"""

print("\nCypher Query:")
print(query3_amount)

print("\nExpected Result:")
print("""
| TotalAmount | NumTransactions |
| ----------- | --------------- |
| 8500        | 3               |
""")

print("\n💡 Advanced Cypher:")
print("  - REDUCE: Accumulate values (like fold/reduce in functional programming)")
print("  - relationships(path): Get all edges")
print("  - SUM: Total amount across all transfers in the path")
print("  - This is fraud detection in action: tracking money flow!")

print("\n✓ Query Type 3 covered: Multi-hop traversal (graph DB advantage)")

---

## 📊 Query Type 4: Aggregation and Graph Statistics

### Use Case: "Who are the high-risk accounts in the network?"

**Why Aggregation Matters:**
- Count transactions per account
- Calculate total amounts
- Identify network hubs (high degree)
- Find risky accounts (many high-value transfers)

**Cypher Aggregation Functions:**
- `COUNT()` - Count items
- `SUM()` - Total amounts
- `AVG()` - Average value
- `MIN()`, `MAX()` - Extremes
- `COLLECT()` - Gather into array
- `GROUP BY` - Implicit grouping in RETURN clause

In [ ]:
# 🔹 Query Type 4: Aggregation and Graph Statistics

print("\n" + "=" * 70)
print("QUERY 4: AGGREGATION - NETWORK STATISTICS")
print("=" * 70)

print("\nKey for Fraud Detection:")
print("  - High transaction volume: Possible money laundering hub")
print("  - High incoming transfers: Suspicious aggregation point")
print("  - High outgoing transfers: Possible dispersal pattern")

# Query 4.1: Transaction volume per account
print("\n\n" + "-" * 70)
print("QUERY 4.1: Transaction Statistics Per Account")
print("(Outgoing transactions)")
print("-" * 70)

query4_volume = """
MATCH (a:Account)-[t:TRANSFERS]->(b:Account)
RETURN a.name as Account, COUNT(t) as OutgoingCount, 
       SUM(t.amount) as TotalOut, AVG(t.amount) as AvgAmount
ORDER BY TotalOut DESC
"""

print("\nCypher Query:")
print(query4_volume)

print("\nExpected Result:")
print("""
| Account | OutgoingCount | TotalOut | AvgAmount |
| ------- | ------------- | -------- | --------- |
| Alice   | 2             | 5500     | 2750      |
| Bob     | 2             | 3000     | 1500      |
| Charlie | 2             | 3500     | 1750      |
| David   | 1             | 3000     | 3000      |
| Frank   | 1             | 950      | 950       |
| Eve     | 0             | 0        | 0         |
""")

print("\n💡 Fraud Detection Signals:")
print("  - Alice: High total volume ($5500 out) → Central to network")
print("  - David: High average amount per transaction ($3000) → Suspicious")
print("  - Eve: No outgoing transactions → Terminal/receiver account")

# Query 4.2: Incoming transactions (receiving)
print("\n\n" + "-" * 70)
print("QUERY 4.2: Who Receives Money? (Incoming Transaction Stats)")
print("-" * 70)

query4_incoming = """
MATCH (a:Account)-[t:TRANSFERS]->(b:Account)
RETURN b.name as Account, COUNT(t) as IncomingCount, 
       SUM(t.amount) as TotalIn, AVG(t.amount) as AvgAmount
ORDER BY TotalIn DESC
"""

print("\nCypher Query:")
print(query4_incoming)

print("\nExpected Result:")
print("""
| Account | IncomingCount | TotalIn | AvgAmount |
| ------- | ------------- | ------- | --------- |
| Charlie | 3             | 5500    | 1833      |
| Alice   | 1             | 950     | 950       |
| Bob     | 1             | 5000    | 5000      |
| David   | 1             | 1500    | 1500      |
| Eve     | 2             | 2500    | 1250      |
| Frank   | 1             | 1000    | 1000      |
""")

print("\n🚨 Red Flags:")
print("  - Charlie: Receiver of $5500 (hub) from 3 accounts → INVESTIGATE")
print("  - Multiple incoming sources → Possible aggregation point for money laundering")

# Query 4.3: Risk score based on activity
print("\n\n" + "-" * 70)
print("QUERY 4.3: Calculate Risk Score (Activity-Based)")
print("-" * 70)

query4_risk = """
MATCH (a:Account)
OPTIONAL MATCH (a)-[out:TRANSFERS]->(o:Account)
OPTIONAL MATCH (i:Account)-[in:TRANSFERS]->(a)
RETURN a.name as Account,
       COUNT(DISTINCT out) as OutgoingTxns,
       COUNT(DISTINCT in) as IncomingTxns,
       COUNT(DISTINCT out) + COUNT(DISTINCT in) as NetworkDegree,
       a.risk as BaseRisk,
       CASE 
         WHEN COUNT(DISTINCT out) + COUNT(DISTINCT in) > 4 THEN 'HIGH'
         WHEN COUNT(DISTINCT out) + COUNT(DISTINCT in) > 2 THEN 'MEDIUM'
         ELSE 'LOW'
       END as ComputedRisk
ORDER BY NetworkDegree DESC
"""

print("\nCypher Query:")
print(query4_risk)

print("\nExpected Result:")
print("""
| Account | OutTxns | InTxns | Degree | BaseRisk | ComputedRisk |
| ------- | ------- | ------ | ------ | -------- | ------------ |
| Alice   | 2       | 1      | 3      | LOW      | MEDIUM       |
| Bob     | 2       | 1      | 3      | MEDIUM   | MEDIUM       |
| Charlie | 2       | 3      | 5      | HIGH     | HIGH         |
| David   | 1       | 1      | 2      | MEDIUM   | LOW          |
| Frank   | 1       | 1      | 2      | MEDIUM   | LOW          |
| Eve     | 0       | 2      | 2      | HIGH     | LOW          |
""")

print("\n💡 Key Concepts:")
print("  - OPTIONAL MATCH: Continue even if pattern not found")
print("  - DISTINCT: Count unique relationships")
print("  - CASE: Conditional logic for risk scoring")

# Query 4.4: Daily transaction statistics
print("\n\n" + "-" * 70)
print("QUERY 4.4: Group by Date - Daily Activity")
print("-" * 70)

query4_daily = """
MATCH (a:Account)-[t:TRANSFERS]->(b:Account)
RETURN t.date as Date, 
       COUNT(t) as DailyTransactions,
       SUM(t.amount) as DailyTotal,
       MIN(t.amount) as MinAmount,
       MAX(t.amount) as MaxAmount
ORDER BY t.date
"""

print("\nCypher Query:")
print(query4_daily)

print("\nExpected Result:")
print("""
| Date       | DailyTxns | DailyTotal | MinAmount | MaxAmount |
| ---------- | --------- | ---------- | --------- | --------- |
| 2024-01-15 | 1         | 5000       | 5000      | 5000      |
| 2024-01-20 | 1         | 2000       | 2000      | 2000      |
| 2024-02-01 | 1         | 1500       | 1500      | 1500      |
| 2024-02-10 | 1         | 500        | 500       | 500       |
| 2024-02-15 | 1         | 1000       | 1000      | 1000      |
| 2024-03-01 | 1         | 950        | 950       | 950       |
| 2024-03-15 | 1         | 3000       | 3000      | 3000      |
| 2024-04-01 | 1         | 2000       | 2000      | 2000      |
""")

print("\n💡 Use Case:")
print("  - Detect spike days (unusual transaction volume)")
print("  - Monitor transaction size patterns")
print("  - Identify timing-based suspicious activity")

print("\n✓ Query Type 4 covered: Aggregation and statistics")

---

## 🔄 Query Type 5: Pattern Detection - Cycles (Highly Suspicious for Fraud!)

### Use Case: "Find money that returns to the source (circular transactions)"

**Why This Matters:**
- **Money laundering indicator:** Money sent out then returned
- **Collusion detection:** Multiple accounts exchanging money in circles
- **Risk assessment:** Circular transactions are highly anomalous

**Cypher Pattern for Cycles:**
```cypher
MATCH (a:Account)-[:TRANSFERS*2..5]->(a)
RETURN a.name as AccountWithCycle
```

**Note:** 
- `(a)-[*2..5]->(a)` → Start and end at same node
- Minimum 2 hops to avoid self-loops
- Maximum 5 hops for performance (can be tuned)

In [ ]:
# 🔹 Query Type 5: Cycle Detection (Suspicious Patterns)

print("\n" + "=" * 70)
print("QUERY 5: CYCLE DETECTION - FIND CIRCULAR TRANSACTIONS")
print("=" * 70)

print("\n🚨 Importance: Cycles indicate potential money laundering!")
print("   Money that returns to source = suspicious circular routing")

# Query 5.1: Simple cycle detection
print("\n\n" + "-" * 70)
print("QUERY 5.1: Find All Cycles (3-5 hops)")
print("-" * 70)

query5_cycles = """
MATCH (a:Account)-[t:TRANSFERS*2..5]->(a)
RETURN a.name as Account, COUNT(t) as PathCount
"""

print("\nCypher Query:")
print(query5_cycles)

print("\nExpected Result:")
print("""
| Account | PathCount |
| ------- | --------- |
| Alice   | 1         |
""")

print("\n💡 Explanation:")
print("  - Alice is part of 1 cycle: Alice → Bob → Frank → Alice")
print("  - CRITICAL: Money sent by Alice returns to Alice")

# Query 5.2: Get the actual cycle path
print("\n\n" + "-" * 70)
print("QUERY 5.2: Show Complete Cycle Paths")
print("-" * 70)

query5_cycle_details = """
MATCH path = (a:Account)-[t:TRANSFERS*2..5]->(a)
WHERE a.id = 'alice'
RETURN nodes(path) as CyclePath, 
       [rel IN relationships(path) | rel.amount] as Amounts,
       REDUCE(sum=0, rel IN relationships(path) | sum + rel.amount) as TotalCycled
"""

print("\nCypher Query:")
print(query5_cycle_details)

print("\nExpected Result:")
print("""
| CyclePath                  | Amounts        | TotalCycled |
| -------------------------- | -------------- | ----------- |
| [Alice, Bob, Frank, Alice] | [5000, 1000, 950] | 6950    |
""")

print("\n🚨 Fraud Insight:")
print("  - Alice sends $5000 to Bob")
print("  - Bob sends $1000 to Frank")
print("  - Frank sends $950 back to Alice")
print("  - Total money cycled: $6950")
print("  - Pattern: Clear circular arrangement (SUSPICIOUS!)")

# Query 5.3: Triangle patterns (3-node cycles)
print("\n\n" + "-" * 70)
print("QUERY 5.3: Triangle Detection (Tightest Cycles)")
print("-" * 70)

query5_triangles = """
MATCH path = (a:Account)-[t1:TRANSFERS]->(b:Account)-[t2:TRANSFERS]->(c:Account)-[t3:TRANSFERS]->(a)
RETURN a.name as Node1, b.name as Node2, c.name as Node3,
       t1.amount as Amount1, t2.amount as Amount2, t3.amount as Amount3
"""

print("\nCypher Query:")
print(query5_triangles)

print("\nExpected Result:")
print("""
| Node1 | Node2 | Node3 | Amount1 | Amount2 | Amount3 |
| ----- | ----- | ----- | ------- | ------- | ------- |
(No results - no 3-node cycles in our network)
""")

print("\n💡 Why Triangles Matter:")
print("  - 3-node cycles are tighter coordination")
print("  - Indicates intentional arrangement")
print("  - Very rare in legitimate transactions")

# Query 5.4: Cycle with high amounts (more suspicious)
print("\n\n" + "-" * 70)
print("QUERY 5.4: High-Value Cycles (> $5000 total)")
print("-" * 70)

query5_high_value = """
MATCH path = (a:Account)-[t:TRANSFERS*2..5]->(a)
WITH path, a, REDUCE(sum=0, rel IN relationships(path) | sum + rel.amount) as cycleAmount
WHERE cycleAmount > 5000
RETURN a.name as Account, cycleAmount as AmountCycled, LENGTH(path) as CycleLength
ORDER BY cycleAmount DESC
"""

print("\nCypher Query:")
print(query5_high_value)

print("\nExpected Result:")
print("""
| Account | AmountCycled | CycleLength |
| ------- | ------------ | ----------- |
| Alice   | 6950         | 3           |
""")

print("\n🚨 Risk Assessment:")
print("  - HIGH-VALUE cycle detected")
print("  - Amount: $6950")
print("  - Action: Escalate for investigation")

# Query 5.5: All accounts in ANY cycle
print("\n\n" + "-" * 70)
print("QUERY 5.5: Identify All Accounts Involved in Cycles")
print("-" * 70)

query5_all_in_cycles = """
MATCH (a:Account)-[t:TRANSFERS*2..5]->(a)
MATCH (a)-[:TRANSFERS]-(neighbor:Account)
RETURN DISTINCT a.name as AccountInCycle, 
       COUNT(DISTINCT neighbor) as ConnectedAccounts
"""

print("\nCypher Query:")
print(query5_all_in_cycles)

print("\nExpected Result:")
print("""
| AccountInCycle | ConnectedAccounts |
| -------------- | ----------------- |
| Alice          | 3                 |
""")

print("\n💡 Takeaway:")
print("  - Alice is the only account in a cycle")
print("  - Alice is connected to 3 other accounts")
print("  - Alice's connectivity + cycle involvement = HIGH RISK")

print("\n✓ Query Type 5 covered: Cycle detection (critical for fraud)")

---

## ⭐ Query Type 6: Hub Detection - Star Patterns (Many Spokes)

### Use Case: "Find accounts that act as money hubs (many connections)"

**Why This Matters:**
- **Money launderer profile:** Hub consolidates and disperses funds
- **Hub account:** Multiple incoming + multiple outgoing
- **Suspicious pattern:** Large volume + many counterparties

**Graph Pattern for Hubs:**
```
      B
      ↓
A → HUB ← C
      ↑
      D
```

Cypher looks for nodes with high degree (many connections)

In [ ]:
# 🔹 Query Type 6: Hub Detection

print("\n" + "=" * 70)
print("QUERY 6: HUB DETECTION - FIND MONEY ROUTING CENTERS")
print("=" * 70)

print("\n🎯 Definition: A hub is an account with high degree")
print("   In/Out degree: How many OTHER accounts connect to/from this account")

# Query 6.1: Find accounts with high total degree
print("\n\n" + "-" * 70)
print("QUERY 6.1: Hub Accounts (High Connectivity)")
print("-" * 70)

query6_hubs = """
MATCH (hub:Account)
OPTIONAL MATCH (hub)-[:TRANSFERS]->(out:Account)
OPTIONAL MATCH (in:Account)-[:TRANSFERS]->(hub)
WITH hub, COUNT(DISTINCT out) as OutDegree, COUNT(DISTINCT in) as InDegree
WHERE OutDegree > 0 OR InDegree > 0
RETURN hub.name as Account, OutDegree, InDegree, 
       (OutDegree + InDegree) as TotalDegree
ORDER BY TotalDegree DESC
"""

print("\nCypher Query:")
print(query6_hubs)

print("\nExpected Result:")
print("""
| Account | OutDegree | InDegree | TotalDegree |
| ------- | ---------- | -------- | ----------- |
| Charlie | 2          | 3        | 5           |
| Alice   | 2          | 1        | 3           |
| Bob     | 2          | 1        | 3           |
| David   | 1          | 1        | 2           |
| Frank   | 1          | 1        | 2           |
| Eve     | 0          | 2        | 2           |
""")

print("\n🚨 Fraud Detection:")
print("  - Charlie: HIGHEST degree (5)")
print("    - Receives from 3 sources")
print("    - Sends to 2 destinations")
print("    - Acts as a CONSOLIDATION POINT (hub)")

# Query 6.2: Hub accounts with incoming + outgoing
print("\n\n" + "-" * 70)
print("QUERY 6.2: True Hub Accounts (Both Incoming AND Outgoing)")
print("-" * 70)

query6_true_hubs = """
MATCH (hub:Account)
MATCH (in:Account)-[tin:TRANSFERS]->(hub)
MATCH (hub)-[tout:TRANSFERS]->(out:Account)
WITH hub, in, out, tin, tout
RETURN DISTINCT hub.name as HubAccount,
       COUNT(DISTINCT in) as IncomingConnections,
       COUNT(DISTINCT out) as OutgoingConnections,
       SUM(tin.amount) as TotalIncoming,
       SUM(tout.amount) as TotalOutgoing
ORDER BY COUNT(DISTINCT in) + COUNT(DISTINCT out) DESC
"""

print("\nCypher Query:")
print(query6_true_hubs)

print("\nExpected Result:")
print("""
| HubAccount | IncomingConnections | OutgoingConnections | TotalIn | TotalOut |
| ---------- | ------------------- | ------------------- | ------- | -------- |
| Charlie    | 3                   | 2                   | 5500    | 3500     |
| Alice      | 1                   | 2                   | 950     | 5500     |
| Bob        | 1                   | 2                   | 5000    | 3000     |
""")

print("\n💡 Money Flow Analysis:")
print("  - Charlie: Receives $5500, sends $3500 (consolidation)")
print("  - Alice: Receives $950, sends $5500 (origination)")
print("  - Bob: Receives $5000, sends $3000 (middle player)")

# Query 6.3: Hub with imbalanced flows (suspicious)
print("\n\n" + "-" * 70)
print("QUERY 6.3: Suspicious Hubs (Large Input Difference)")
print("-" * 70)

query6_suspicious = """
MATCH (hub:Account)
OPTIONAL MATCH (in:Account)-[tin:TRANSFERS]->(hub)
OPTIONAL MATCH (hub)-[tout:TRANSFERS]->(out:Account)
WITH hub, SUM(tin.amount) as totalIn, SUM(tout.amount) as totalOut
WHERE totalIn IS NOT NULL AND totalOut IS NOT NULL
WITH hub, totalIn, totalOut, (totalIn - totalOut) as Difference
WHERE ABS(Difference) > 1000
RETURN hub.name as Account, totalIn, totalOut, Difference, 
       ROUND((Difference / totalIn) * 100) as DiscrepancyPercent
ORDER BY ABS(Difference) DESC
"""

print("\nCypher Query:")
print(query6_suspicious)

print("\nExpected Result:")
print("""
| Account | TotalIn | TotalOut | Difference | DiscrepancyPercent |
| ------- | ------- | -------- | ---------- | ------------------ |
| Charlie | 5500    | 3500     | 2000       | 36                 |
| Alice   | 950     | 5500     | -4550      | -479               |
| Bob     | 5000    | 3000     | 2000       | 40                 |
""")

print("\n🚨 Red Flags:")
print("  - Charlie: Receives $5500, sends only $3500 → $2000 unaccounted (36% loss)")
print("  - Alice: Sends $5500 but receives only $950 → $4550 net outflow (-479%)")
print("  - Pattern: Money enters Charlie, partially exits → SUSPICIOUS")

# Query 6.4: Star pattern visualization
print("\n\n" + "-" * 70)
print("QUERY 6.4: Get Complete Star Pattern Around Charlie")
print("-" * 70)

query6_star = """
MATCH (incoming:Account)-[in_rel:TRANSFERS]->(hub:Account)
WHERE hub.name = 'Charlie'
MATCH (hub)-[out_rel:TRANSFERS]->(outgoing:Account)
RETURN incoming.name as Sender, hub.name as Hub, outgoing.name as Receiver,
       in_rel.amount as InAmount, out_rel.amount as OutAmount
"""

print("\nCypher Query:")
print(query6_star)

print("\nExpected Result:")
print("""
| Sender   | Hub     | Receiver | InAmount | OutAmount |
| -------- | ------- | -------- | -------- | --------- |
| Alice    | Charlie | David    | 5000     | 1500      |
| Alice    | Charlie | Eve      | 5000     | 2000      |
| Bob      | Charlie | David    | 2000     | 1500      |
| Bob      | Charlie | Eve      | 2000     | 2000      |
| David    | Charlie | David    | 3000     | 1500      |
| David    | Charlie | Eve      | 3000     | 2000      |
""")

print("\n💡 Note:")
print("  - Each incoming connection combines with each outgoing")
print("  - Creates a Cartesian product (all combinations)")
print("  - Filter to get specific hubs if needed")

print("\n✓ Query Type 6 covered: Hub detection")

---

## 🔬 Query Type 7: Advanced Analysis - Complete Fraud Scoring

### Use Case: "Create a comprehensive fraud risk score combining all patterns"

**Combines:**
- Hub activity (high degree)
- Cycle involvement (circular patterns)
- Volume anomalies (unusually high amounts)
- Network depth (reach in graph)

In [ ]:
# 🔹 Query Type 7: Advanced Fraud Risk Scoring

print("\n" + "=" * 70)
print("QUERY 7: ADVANCED ANALYSIS - COMPREHENSIVE FRAUD SCORING")
print("=" * 70)

print("\nCombines multiple fraud signals:")
print("  1. Hub activity (degree centrality)")
print("  2. Cycle involvement (circular patterns)")
print("  3. Volume abnormalities")
print("  4. Network position")

# Query 7.1: Complete risk assessment
print("\n\n" + "-" * 70)
print("QUERY 7.1: Comprehensive Fraud Risk Score")
print("-" * 70)

query7_risk = """
WITH 
  // Identify hub accounts
  (MATCH (a:Account)
   OPTIONAL MATCH (a)-[:TRANSFERS]->(out1:Account)
   OPTIONAL MATCH (in1:Account)-[:TRANSFERS]->(a)
   RETURN a, COUNT(DISTINCT out1) + COUNT(DISTINCT in1) as degree) as hubs,
  
  // Identify cycle accounts
  (MATCH (c:Account)-[:TRANSFERS*2..5]->(c)
   RETURN DISTINCT c as inCycle) as cycles

MATCH (account:Account)
WHERE account IN [h.a | h in hubs]

OPTIONAL MATCH (account)-[out:TRANSFERS]->(outbound:Account)
OPTIONAL MATCH (inbound:Account)-[in:TRANSFERS]->(account)

WITH account,
     COUNT(DISTINCT out) as outDegree,
     COUNT(DISTINCT in) as inDegree,
     SUM(out.amount) as totalOut,
     SUM(in.amount) as totalIn,
     CASE WHEN account IN [c.inCycle | c IN cycles] THEN 1 ELSE 0 END as inCycle

RETURN account.name as Account,
       inDegree + outDegree as NetworkDegree,
       ROUND(((inDegree + outDegree) / 6.0) * 40) as HubScore,
       inCycle * 30 as CycleScore,
       ROUND((COALESCE(totalIn, 0) + COALESCE(totalOut, 0)) / 100000 * 20) as VolumeScore,
       ROUND(((inDegree + outDegree) / 6.0) * 40) + (inCycle * 30) + 
       ROUND((COALESCE(totalIn, 0) + COALESCE(totalOut, 0)) / 100000 * 20) + 10 as TotalScore
ORDER BY TotalScore DESC
"""

print("\nCypher Query (Simplified for readability):")
print("""
// Complex multi-stage query combining:
// 1. Hub detection (degree analysis)
// 2. Cycle detection  
// 3. Volume analysis
// 4. Scoring aggregation
""")

print("\nExpected Result (Risk Scores 0-100):")
print("""
| Account | Degree | HubScore | CycleScore | VolumeScore | TotalScore |
| ------- | ------ | -------- | ---------- | ----------- | ---------- |
| Charlie | 5      | 34       | 0          | 7           | 41         |
| Alice   | 3      | 20       | 30         | 6           | 56         |
| Bob     | 3      | 20       | 0          | 5           | 25         |
| David   | 2      | 13       | 0          | 3           | 16         |
| Frank   | 2      | 13       | 0          | 1           | 14         |
| Eve     | 2      | 13       | 0          | 2           | 15         |
""")

print("\n🚨 FRAUD RISK ASSESSMENT:")
print("  - Alice (Score: 56): HIGHEST RISK")
print("    → Part of cycle (30 points)")
print("    → High degree (20 points)")
print("    → Significant volume (6 points)")
print()
print("  - Charlie (Score: 41): HIGH RISK")
print("    → Hub account (34 points)")
print("    → High volume (7 points)")
print("    → Not in cycle (might be money launderer)")
print()
print("  - Bob (Score: 25): MEDIUM RISK")
print("    → Moderate hub activity")
print()
print("  - Others: LOWER RISK")

# Query 7.2: Investigation query - detailed account breakdown
print("\n\n" + "-" * 70)
print("QUERY 7.2: Detailed Investigation for High-Risk Accounts")
print("-" * 70)

query7_investigation = """
MATCH (account:Account {id: 'alice'})
MATCH (account)-[out:TRANSFERS]->(out_acct:Account)
MATCH (in_acct:Account)-[in:TRANSFERS]->(account)
MATCH path = (account)-[:TRANSFERS*1..3]->(dest:Account)

RETURN 
  account.name as Account,
  account.balance as Balance,
  account.risk as BaseRisk,
  COUNT(DISTINCT out_acct) as OutgoingTo,
  COUNT(DISTINCT in_acct) as IncomingFrom,
  SUM(out.amount) as TotalOutgoing,
  SUM(in.amount) as TotalIncoming,
  COUNT(DISTINCT dest) as Reach3Hops,
  LENGTH(path) as PathLength
"""

print("\nCypher Query:")
print(query7_investigation)

print("\nExpected Result:")
print("""
| Account | Balance | BaseRisk | OutTo | InFrom | TotalOut | TotalIn | Reach | PathLen |
| ------- | ------- | -------- | ----- | ------ | -------- | ------- | ----- | ------- |
| Alice   | 10000   | LOW      | 2     | 1      | 5500     | 950     | 5     | 3       |
""")

print("\n💡 Investigation Talking Points:")
print("  - Account: Alice (LOW base risk, but high transaction activity)")
print("  - Balance: $10,000 (reasonable for activity level)")
print("  - Outgoing: $5500 to 2 people (suspicious - high amount)")
print("  - Incoming: $950 from 1 person (circular money)")
print("  - Reach: Can contact 5 other accounts in 3 hops (central in network)")
print("  - Action: Escalate to compliance team for detailed review")

print("\n✓ Query Type 7 covered: Advanced fraud scoring")

# 🎓 Enhanced Summary: Mastering Cypher for Fraud Detection

## 📋 Complete Query Reference

### Query Type Summary

| Query Type | Purpose | Complexity | Performance | Fraud Use |
| --- | --- | --- | --- | --- |
| **1. CREATE** | Build graph | Low | Fast | Setup only |
| **2. MATCH** | Simple lookups | Low | ⚡⚡⚡ Microseconds | Baseline queries |
| **3. Multi-hop** | Trace chains | Medium | ⚡⚡ Milliseconds | Money trail tracking |
| **4. Aggregation** | Stats/scoring | Medium | ⚡⚡ Milliseconds | Risk calculation |
| **5. Cycles** | Circular patterns | High | ⚡ Seconds | Key fraud indicator |
| **6. Hubs** | High-degree nodes | High | ⚡ Seconds | Money laundering detection |
| **7. Risk Score** | Combined analysis | Very High | ⚡ Seconds | Comprehensive assessment |

---

## 🔧 Cypher Quick Reference

### Core Patterns

**Simple Connection:**
```cypher
MATCH (a:Label)-[:RELATIONSHIP]->(b:Label)
RETURN a, b
```

**Multi-Hop Path:**
```cypher
MATCH (a)-[:RELATIONSHIP*1..3]->(b)
RETURN a, b
```

**Cycle Detection:**
```cypher
MATCH (a)-[:RELATIONSHIP*2..5]->(a)
RETURN a
```

**Hub Detection:**
```cypher
MATCH (hub)-[:RELATIONSHIP]-(neighbor)
RETURN hub, COUNT(DISTINCT neighbor) as Degree
```

**Aggregation:**
```cypher
RETURN node.property, COUNT(*) as Count, SUM(rel.value) as Total
```

### Key Cypher Functions

| Function | Purpose | Example |
| --- | --- | --- |
| `MATCH` | Find patterns | `MATCH (a)-[:REL]->(b)` |
| `WHERE` | Filter results | `WHERE a.amount > 1000` |
| `RETURN` | Output format | `RETURN a.name, b.name` |
| `COUNT()` | Count items | `COUNT(DISTINCT nodes)` |
| `SUM()` | Total values | `SUM(relationship.amount)` |
| `AVG()`, `MIN()`, `MAX()` | Statistics | `AVG(rel.amount)` |
| `COLLECT()` | Gather into list | `COLLECT(node.id)` |
| `REDUCE()` | Accumulate value | `REDUCE(sum=0, x IN list\|sum+x.val)` |
| `nodes(path)` | Extract nodes | `nodes(path)` |
| `relationships(path)` | Extract relationships | `relationships(path)` |
| `LENGTH(path)` | Count hops | `LENGTH(path)` |
| `DISTINCT` | Remove duplicates | `RETURN DISTINCT account` |
| `ORDER BY` | Sort results | `ORDER BY amount DESC` |
| `LIMIT` | Top N results | `LIMIT 10` |
| `CASE` | Conditional logic | `CASE WHEN x > 10 THEN 'HIGH' ELSE 'LOW' END` |
| `WITH` | Chain operations | `WITH result WITH ... MATCH ...` |
| `OPTIONAL MATCH` | Optional patterns | `OPTIONAL MATCH (a)-[:X]->(b)` |

---

## ⚡ Performance Optimization Tips

### 1. Indexing (Critical for Large Graphs)

```cypher
-- Create index on frequently searched properties
CREATE INDEX ON :Account(id)
CREATE INDEX ON :Account(risk)
```

**When to use:**
- Properties used in WHERE clauses
- Labels searched frequently
- Relationship properties used in filters

### 2. Limit Hops (Prevents Explosive Expansion)

❌ Bad (can timeout):
```cypher
MATCH (a)-[:TRANSFERS*]->(b)  -- Unlimited hops!
```

✅ Good:
```cypher
MATCH (a)-[:TRANSFERS*1..3]->(b)  -- Limited to 3 hops
```

### 3. Use OPTIONAL MATCH Carefully

❌ Inefficient:
```cypher
MATCH (a:Account)
MATCH (a)-[:TRANSFERS]->(b:Account)
```

✅ Better:
```cypher
MATCH (a:Account)
OPTIONAL MATCH (a)-[:TRANSFERS]->(b:Account)
```

### 4. Filter Early (Reduce Working Set)

✅ Better (filter first):
```cypher
MATCH (a:Account {risk: 'HIGH'})
MATCH (a)-[:TRANSFERS]->(b)
RETURN a, b
```

❌ Worse (filter last):
```cypher
MATCH (a:Account)
MATCH (a)-[:TRANSFERS]->(b)
WHERE a.risk = 'HIGH'
RETURN a, b
```

### 5. Use DISTINCT Wisely

❌ Bad (counts duplicates):
```cypher
RETURN paths
```

✅ Good (removes duplicates):
```cypher
RETURN DISTINCT paths
```

### 6. Avoid Cartesian Products

❌ Creates explosion:
```cypher
MATCH (a:Account)-[:TRANSFERS]->(b:Account)
MATCH (c:Account)-[:TRANSFERS]->(d:Account)
-- Without connection between b and c: (a,b,c,d) combinations
```

✅ Connected properly:
```cypher
MATCH (a:Account)-[:TRANSFERS]->(b:Account)-[:TRANSFERS]->(c:Account)
-- Follows chain, no explosion
```

---

## 🚀 Complete Fraud Detection Workflow (in Production)

### 1️⃣ Real-Time Detection

```cypher
-- When new transaction arrives:
MATCH (a:Account {id: 'new_sender'})
MATCH (b:Account {id: 'new_receiver'})
WHERE a.id <> b.id

-- Check if creates cycle
OPTIONAL MATCH cycle = (a)-[:TRANSFERS*2..5]->(a)
WITH a, b, cycle

-- Check if hub involvement
OPTIONAL MATCH (hub:Account)
WHERE hub.risk = 'HIGH'
AND ((a)-[:TRANSFERS*1..2]-(hub) OR (hub)-[:TRANSFERS*1..2]-(b))

-- Return risk signal
RETURN 
  a.name as Sender,
  b.name as Receiver,
  CASE WHEN cycle IS NOT NULL THEN 'CYCLE' ELSE 'OK' END as CycleCheck,
  CASE WHEN hub IS NOT NULL THEN hub.name ELSE 'NONE' END as HubInvolved,
  'ALERT_NEEDED' as Action
```

### 2️⃣ Batch Risk Scoring

```cypher
-- Run nightly to update all accounts
MATCH (a:Account)
WITH a
OPTIONAL MATCH (a)-[out:TRANSFERS]->(out_acct:Account)
OPTIONAL MATCH (in_acct:Account)-[in:TRANSFERS]->(a)
OPTIONAL MATCH cycle = (a)-[:TRANSFERS*2..5]->(a)

WITH a, COUNT(DISTINCT out_acct) + COUNT(DISTINCT in_acct) as degree,
     SUM(out.amount) + SUM(in.amount) as volume,
     CASE WHEN cycle IS NOT NULL THEN 1 ELSE 0 END as has_cycle

SET a.computed_risk_score = degree * 10 + volume * 0.001 + has_cycle * 50
```

### 3️⃣ Investigation Query

```cypher
-- Analyst investigating Alice
MATCH (a:Account {name: 'Alice'})

-- Get transaction neighbors
MATCH (a)-[t:TRANSFERS]-(neighbor:Account)

-- Get paths to other high-risk accounts
OPTIONAL MATCH path = (a)-[:TRANSFERS*1..4]-(risky:Account {risk: 'HIGH'})

-- Get account stats
WITH a, 
     COUNT(DISTINCT neighbor) as neighbors,
     COUNT(DISTINCT risky) as risky_paths,
     COLLECT({account: neighbor.name, amount: t.amount}) as transactions

RETURN {
  account: a.name,
  balance: a.balance,
  risk_score: a.risk,
  neighbors: neighbors,
  risky_connections: risky_paths,
  transactions: transactions
}
```

---

## 📊 Comparison: Cypher vs SQL

### Query: "Find accounts that transfer money to accounts with HIGH risk"

**SQL (Relational Database):**
```sql
SELECT DISTINCT a1.name as Sender, a2.name as Receiver, t.amount
FROM Accounts a1
JOIN Transactions t ON a1.id = t.from_id
JOIN Accounts a2 ON a2.id = t.to_id
WHERE a2.risk = 'HIGH'
ORDER BY t.amount DESC
LIMIT 100;
```

**Cypher (Graph Database):**
```cypher
MATCH (a1:Account)-[t:TRANSFERS]->(a2:Account {risk: 'HIGH'})
RETURN DISTINCT a1.name as Sender, a2.name as Receiver, t.amount
ORDER BY t.amount DESC
LIMIT 100
```

**Advantages of Cypher:**
- ✅ Simpler syntax (follows graph structure)
- ✅ No explicit JOINs needed (relationships are first-class)
- ✅ More efficient for multi-hop queries
- ✅ Natural pattern matching

---

## 🔐 Security Best Practices

### 1. Parameter Binding (Prevents Cypher Injection)

❌ Vulnerable:
```cypher
MATCH (a:Account) WHERE a.name = $username
RETURN a
```

✅ Safe:
```cypher
MATCH (a:Account {name: $username})
RETURN a
```

### 2. Role-Based Access Control

```cypher
-- Neo4j Enterprise: Define roles
CREATE ROLE investigator;
GRANT READ ON GRAPH fraud_network TO investigator;
REVOKE DELETE ON GRAPH fraud_network FROM investigator;
```

### 3. Audit Logging

```cypher
-- Log all queries for compliance
CREATE (log:AuditLog {
  query: $query,
  user: $user,
  timestamp: datetime(),
  result_count: $count
})
```

---

## 🧪 Testing Cypher Queries

### Development Workflow

1. **Write Query**
   ```cypher
   MATCH (a:Account)-[:TRANSFERS]->(b:Account)
   RETURN a.name, b.name LIMIT 10
   ```

2. **Check Execution Plan**
   ```cypher
   EXPLAIN MATCH (a:Account)-[:TRANSFERS]->(b:Account)
   RETURN a.name, b.name
   ```

3. **Profile for Performance**
   ```cypher
   PROFILE MATCH (a:Account)-[:TRANSFERS]->(b:Account)
   RETURN a.name, b.name
   ```

4. **Optimize Indexes**
   ```cypher
   CREATE INDEX idx_account_id ON :Account(id)
   ```

---

## 📚 Advanced Concepts

### 1. APOC (Awesome Procedures on Cypher)

Extended library of useful procedures:

```cypher
-- Shortest path with weight
MATCH (a:Account {id: 'alice'}), (b:Account {id: 'david'})
CALL apoc.algo.dijkstra(a, b, 'TRANSFERS>', 'amount') YIELD path, weight
RETURN path, weight
```

### 2. Graph Algorithms (Neo4j GDS)

```cypher
-- PageRank for account influence
CALL gds.pageRank.stream('transactionGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name as Account, score
ORDER BY score DESC
```

### 3. Subgraph Projection

```cypher
-- Analyze only high-value transfers
CALL gds.graph.project(
  'subgraph',
  'Account',
  {TRANSFERS: {properties: 'amount'}},
  {nodeProperties: 'risk'}
)
```

---

## 🎯 Real-World Deployment Checklist

- [ ] Create indexes on frequently searched properties
- [ ] Set up connection pooling for applications
- [ ] Enable authentication and role-based access
- [ ] Configure audit logging
- [ ] Set up monitoring and alerting
- [ ] Test query performance with large datasets
- [ ] Document Cypher query library for team
- [ ] Implement caching for frequent queries
- [ ] Set up regular backups
- [ ] Plan for scaling (clustering, sharding)

---

## ❓ Common Questions

**Q: How large can graphs get?**
A: Neo4j handles billions of nodes/relationships. Performance depends on query complexity and indexing.

**Q: How do I join multiple graphs?**
A: Neo4j is a single graph. Use multiple databases (Neo4j Enterprise) or federate externally.

**Q: Can I do transactions in Cypher?**
A: Yes, within a single query context. Multi-query transactions need explicit management.

**Q: How do I export results?**
A: Use `APOC` library for exports. Results can be streamed to CSV, JSON, etc.

**Q: Performance degrading - what to check?**
A: 1) Check indexes 2) Use EXPLAIN/PROFILE 3) Optimize hops 4) Monitor memory/CPU

---

## 🚀 Integration with ML Pipeline

### Feature Extraction with Cypher

```cypher
-- Extract features for ML model
MATCH (a:Account)-[t:TRANSFERS*1..3]->(b:Account)
RETURN a.id,
  COUNT(DISTINCT t) as transaction_count,
  SUM(t.amount) as total_amount,
  AVG(t.amount) as avg_amount,
  a.balance as account_balance,
  a.risk as label
LIMIT 1000
```

### Export to CSV for Model Training

```cypher
CALL apoc.export.csv.query(
  'MATCH (a)-[t:TRANSFERS]-(b) RETURN a.id, b.id, t.amount',
  'transactions.csv',
  {}
)
```

---

## 🔄 What's Next in Your Fraud Detection Journey

**This Notebook (10):** Graph Database Fundamentals + Cypher Queries

**Next Steps:**
1. **Notebook 11 (Neptune Analytics):** Cloud-scale graph analytics on AWS
2. **Notebook 12 (Capstone Project):** End-to-end fraud detection system combining all notebooks
3. **Advanced Topics:**
   - Graph Neural Networks (GNNs)
   - Temporal graph analysis (transactions over time)
   - Knowledge graphs (integrating external fraud databases)
   - Real-time streaming graph updates

---

## 📝 Key Takeaways

✅ **Graph Databases Excel at Relationships:** Multi-hop queries that are slow in SQL are fast in Neo4j

✅ **Cypher is Intuitive:** Pattern syntax matches graph structure visually

✅ **Fraud Patterns are Graph Problems:** Cycles, hubs, and paths are perfect for graph analysis

✅ **Combine Multiple Signals:** Hub + cycle + volume detection beats single measures

✅ **Index and Profile:** Performance optimization is critical for production systems

✅ **Neo4j Production Ready:** Used by major banks and fintech for real fraud detection

---

## 📚 Complete Fraud Detection Query Library

You now have 7 core query patterns. Combine them based on your needs:

| Pattern | Query | Use Case |
| --- | --- | --- |
| 1. Direct | MATCH (a)-[:REL]->(b) | Simple connections |
| 2. Filtered | MATCH...WHERE... | With conditions |
| 3. Multi-hop | MATCH (a)-[:REL*1..3]->(b) | Money trails |
| 4. Aggregation | ...GROUP BY...SUM()... | Statistics |
| 5. Cycles | MATCH (a)-[:REL*2..5]->(a) | Circular patterns |
| 6. Hubs | COUNT(connected nodes) | High degree |
| 7. Risk Score | Combined metrics | Production scoring |

**Practice:** Combine these patterns to solve increasingly complex fraud scenarios!

---

## 🎓 Interview Preparation

**Common Questions:**
1. "Why use Neo4j over SQL for fraud detection?" → Relationship-heavy, multi-hop queries, pattern matching
2. "How do cycles indicate fraud?" → Circular money flow = money laundering
3. "What's index-free adjacency?" → Direct pointers between nodes = O(1) traversal
4. "How to optimize Cypher queries?" → Indexes, EXPLAIN/PROFILE, limit hops, filter early
5. "Explain the difference between Cypher and SQL JOINS" → Cypher uses relationships (first-class), SQL requires explicit JOINs

---

## 📞 Support Resources

- **Neo4j Documentation:** https://neo4j.com/docs/
- **Cypher Manual:** https://neo4j.com/docs/cypher-manual/
- **Neo4j Community:** https://community.neo4j.com/
- **Graph Algorithms:** https://neo4j.com/docs/graph-data-science/
- **APOC Library:** https://neo4j.com/docs/apoc/current/